# Understanding VAE

## 1. Pre-requisites

### 1a. Convexity and Concavity

A function $f$ is strictly convex if, for any two points on its graph, the straight line connecting them (also called a chord) lies on or above the function graph. The local minimum of a strictly convex function is automatically its global minimum. A function $f$ is strictly concave if the chord instead lies on or below the function. The local maximum of a strictly concave function is automatically its global maximum.

![A strictly convex and a strictly concave function, each with a chord](graphs/convexity_examples.png)

### 1b. Jensen's Inequality

We can show that for a **concave** function $\varphi$ and random variable $X$:

$$\varphi(\mathbb{E}[X]) \;\geq\; \mathbb{E}[\varphi(X)]$$

If $\varphi=\log$ function (which is concave), and we sample $n$ times from $X$, this means:

$$\log\big(\text{average}(x)\big) \;\geq\; \text{average}\big(\log(x)\big)$$

![Jensen's inequality for the concave function ln(x)](graphs/jensen_general.png)

### 1c. KL Divergence

Consider **Shannon Entropy** $H(P)$ where $x \sim P(x)$. The $\frac{1}{P(x_i)}$ quatifies *suprise*, and so $H(P)$ is an expected suprise given that $X$ is drawn from distribution $P$.

$$H(P) = \sum_{i=1}^{n} P(x_i) \log \left( \frac{1}{P(x_i)} \right) = \mathbb{E}_{P(x)}\!\left[\log \frac{1}{P(x)}\right]$$

Let us now consider some unknown distribution $Q(x)$ so that

$$\sum_{i=1}^{n} P(x_i) \log \left( \frac{1}{Q(x_i)} \right)$$

Now here is a question, what distribution $Q(x)$ minimises the expected suprise? We can prove that the lowest possible value the expected suprise achieves is when $Q(x)=P(x)$. For a simple binary case, we can prove that with simple algebra:

$$\begin{aligned}
H(q) &= p \log \left( \frac{1}{q} \right) + (1-p) \log \left( \frac{1}{1-q} \right) \\[6pt]
&= -p \ln(q) - (1-p) \ln(1-q)
\end{aligned}$$

We know that a $log$ function is concave, so $-log$ has to be convex, meaning $H(q)$ has a global mininum 
$$\begin{aligned}
\frac{dH(q)}{dq} &= -p \ln(q) - (1-p) \ln(1-q) \\[6pt]
&= -\frac{p}{q} - (1-p) \cdot \left( \frac{-1}{1-q} \right) \\[6pt]
&= -\frac{p}{q} + \frac{1-p}{1-q}
\end{aligned}$$

The mininum value of $H(q)$ is at the point $q$ where $\frac{dH(q)}{dq}=0$
$$\begin{aligned}
-\frac{p}{q} + \frac{1-p}{1-q} &= 0 \\[6pt]
\frac{1-p}{1-q} &= \frac{p}{q} \\[6pt]
q(1-p) &= p(1-q) \\[6pt]
q - pq &= p - pq \\[6pt]
q &= p
\end{aligned}$$

The above is called **Cross-Entropy** $H(P, Q)$. Given that we know $Q$ cannot yield lower entropy than $P$, we can think of
$H(P, Q)$ as an expected suprise under a wrong belief.

$$H(P, Q)=\sum_{i=1}^{n} P(x_i) \log \left( \frac{1}{Q(x_i)} \right)$$

Now, let us consider this

$$\begin{aligned}
H(P, Q) - H(P) &= -\sum_{x} P(x) \log Q(x) - \left( -\sum_{x} P(x) \log P(x) \right) \\[6pt]
&= -\sum_{x} P(x) \log Q(x) + \sum_{x} P(x) \log P(x) \\[6pt]
&= \sum_{x} P(x) \Big( \log P(x) - \log Q(x) \Big) \\[6pt]
&= \sum_{x} P(x) \log \left( \frac{P(x)}{Q(x)} \right) \\[6pt]
&= D_{\text{KL}}(P \parallel Q)
\end{aligned}$$

The $D_{\text{KL}}(P \parallel Q)$ is known as the **Kullback-Leibler (KL) divergence** and it measures the extra cost of using a wrong belief. Also commonly wrriten as $\text{KL}(P \parallel Q)$. Same equation holds for the continous case:

$$\text{KL}(p \,\|\, q) \;=\; \mathbb{E}_{p}\!\left[\log \frac{p(x)}{q(x)}\right] \;=\; \int p(x) \log \frac{p(x)}{q(x)} \, dx$$

Important properties of KL divergence we'll need:
- **Non-negative**: $\text{KL}(p\|q) \geq 0$. We already showed that $H(P,Q) \geq H(P)$ for a binary distribution, but the general case can be proved like so:

$$
\begin{aligned}
\text{KL}(p \parallel q) &= \mathbb{E}_p\!\left[\log \frac{p(x)}{q(x)}\right] \\[4pt]
&= -\,\mathbb{E}_p\!\left[\log \frac{q(x)}{p(x)}\right]
\end{aligned}
$$

$$
\begin{aligned}
\mathbb{E}_p\!\left[\log \frac{q(x)}{p(x)}\right] &\leq \log \, \mathbb{E}_p\!\left[\frac{q(x)}{p(x)}\right] \\[4pt]
-\mathbb{E}_p\!\left[\log \frac{q(x)}{p(x)}\right] &\geq -\log \, \mathbb{E}_p\!\left[\frac{q(x)}{p(x)}\right] \\[4pt]
&\geq -\log \int p(x)\,\frac{q(x)}{p(x)}\,dx \\[4pt]
&\geq -\log \int q(x)\,dx \\[4pt]
&\geq -\log 1 \\[4pt]
&\geq 0 \\[4pt]
\text{KL}(p \parallel q) &\geq 0
\end{aligned}
$$

- It is **asymmetric**, $\text{KL}(p \parallel q) \neq \text{KL}(q \parallel p)$. Suppose $p(x)$, a mixture of Gaussians, is the true distribution, and $q(x)$, a regular Gaussian, is the approximate distribution produced by the model.
    - **Forward** $\text{KL}(p \parallel q)=\mathbb{E}_p\!\left[\log \frac{p(x)}{q(x)}\right]$ - if $p(x) > 0$ and $q(x) \to 0$, the $\log \left( \frac{p(x)}{q(x)} \right) \to \infty$. To minimise $KL(p \parallel q)$, the model $q(x)$ is incentivized to be $q(x) > 0$ whenever $p(x) > 0$. As a result, $q(x)$ spreads out over $p$ to cover as many positive $p(x)$ as possible.
    - **Reverse** $\text{KL}(q \parallel p)=\mathbb{E}_q\!\left[\log \frac{q(x)}{p(x)}\right]$ - if $q(x) > 0$ and $p(x) \to 0$, the $\log \left( \frac{q(x)}{p(x)} \right) \to \infty$. The model $q(x)$ is penalised the most when assigning high likelihood, whenever $p(x)$ assigns low likelihood. As a result, $q(x)$ prioritizes regions where $p(x)$ is high, assigning low likelihood to anything else. 

![Forward vs reverse KL fits of a Gaussian to a bimodal distribution](graphs/kl_asymmetry.png)

## 2. Problem Statement

### 2a. Optimization objective

Consider a dataset $\mathcal{D}$ with $n$ independent and identically distributed examples, $\mathcal{D} = \{\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_n\}$, where $\mathbf{x} \in \mathbb{R}^d$. We would like to learn a model $p_{\boldsymbol{\theta}}(\mathbf{x})$, parametrized by $\boldsymbol{\theta}$, that assings high probability/likelihood to every example from $\mathcal{D}$. In theory, if $p_{\boldsymbol{\theta}}(\mathbf{x})$ can do that, it means it captured (or at least approximated), the true distribution $p_*(\mathbf{x})$ from which $\mathbf{x}$ was sampled. Our goal is therefore simple, maximise $\prod_{i=1}^{n} p_{\boldsymbol{\theta}}(\mathbf{x}_i)$. However, if we deal with probabilities, the product ends up being a very small number close to $0$, which in practice, makes it difficult to train the model. To ease training, we convert the product to a sum via the $\log$ function. So the parameters of our model are the ones that maximise

$$\hat{\boldsymbol{\theta}} = \arg\max_{\boldsymbol{\theta}} \sum_{i=1}^{n} \log p_{\boldsymbol{\theta}}(\mathbf{x}_i)$$

Once we have $p_{\hat{\boldsymbol{\theta}}}(\mathbf{x})$, we can:
 - reason about any new example $\mathbf{x}_{n+1}$. If $p_{\hat{\boldsymbol{\theta}}}(\mathbf{x}_{n+1})$ is high, the new example is likely also drawn from $p_*(\mathbf{x})$ and is similar to examples from $\mathcal{D}$. And the reverse, if $p_{\hat{\boldsymbol{\theta}}}(\mathbf{x}_{n+1})$ is low, $\mathbf{x}_{n+1}$ is likely from a different distribution and is very different to our dataset $\mathcal{D}$.
 - sample brand new examples $\mathbf{x}$, that were not part of $\mathcal{D}$, but for which $p_{\hat{\boldsymbol{\theta}}}(\mathbf{x})$ is high.

### 2b. Regular Autoencoder

How do we actually model $p_{\boldsymbol{\theta}}(\mathbf{x})$ in practice though? Let's consider a simple approach first, making $p_{\boldsymbol{\theta}}(\mathbf{x})$ a single neural network. We feed $\mathbf{x}$ as the input, $\mathbf{z} \in \mathbb{R}^m$ is the lower dimensional latent space ($d > m$), and the output layer produces a reconstruction $\mathbf{\hat{x}}$. So, we are actually maximising $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$. Not exactly what we want, but it's pretty straightforward, so let's give it a try.

Given that $\boldsymbol{\theta} = \{\boldsymbol{\phi}, \boldsymbol{\gamma}\}$, we can think about learning two parts of the network:
- a deterministic encoder $\mathbf{z} = f_{\boldsymbol{\phi}}(\mathbf{x})$
- a deterministic decoder $\mathbf{\hat{x}} = g_{\boldsymbol{\gamma}}(\mathbf{z})$

trained to minimize a reconstruction loss between $\mathbf{x}$ and $\mathbf{\hat{x}}$ (e.g. MSE, Cross-Entropy etc). We can learn to reconstruct $\mathcal{D}$, but the problem is nothing in our training objective constrains where encoded points $\mathbf{z}$ land in latent space. As a result:

- the encoder is free to scatter training points into arbitrary, disconnected regions of the latent space
- to generate new $\mathbf{x}$ we would need to sample a "plausible" $\mathbf{z}$ and decode it. But we don't know which $\mathbf{z}$ are "plausible", so the best we can do is sample random $\mathbf{z}$ and hope for the best
- because $\mathcal{D}$ is encoded into disconnected regions of the latent space, most random $\mathbf{z}$ we try, will produce a garbage output because the decoder has never seen them during training.
- $\mathbf{z}$ that produce similar $\mathbf{\hat{x}}$, are not close to each other in the latent space

That means a regular autoencoder learns a good reconstruction map, but, as expected, it completely fails as an approximation of $p_*(x)$, which means we cannot use it as a generative model $p_{\boldsymbol{\theta}}(\mathbf{x})$ from which we can sample $\mathbf{x}$. All we can do is sample $\mathbf{x}$ from $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$, but because $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$ is a deterministic decoder, it learnt a $1:1$ mapping instead of a distribution over $\mathbf{x}$, so it will always produce the same reconstruction. The model simply finds a $\mathbf{z}$ for which $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$ is the highest.

Because the latent space $\mathbf{z}$ is completely unconstrained, and thus, the encoder scatters representations of $\mathbf{x}$ anywhere it wants creating isolated islands that occupy a near-zero volume in $\mathbb{R}^m$, this means for almost any randomly chosen $\mathbf{z}$, the decoder will output garbage. Integrating over this unconstrained space yields a marginal likelihood of effectively zero:

$$p_{\boldsymbol{\theta}}(\mathbf{x}) = \int_{\mathbb{R}^m} p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})\, p(\mathbf{z}) \, d\mathbf{z} \approx 0$$

### 2c. Variational Autoencoder

It's clear from the regular autoencoder case, that in order to maximise $p_{\boldsymbol{\theta}}(\mathbf{x})$, we have to force the encoder to learn which region(s) of the latent space, instead of a single value of $\mathbf{z}$, yield good reconstructions. By introducing a prior distribution $p(\mathbf{z})$ (e.g. $\mathcal{N}(0, I)$), we further enforce the structure of these region(s). We can now evaluate how good the parameters $\boldsymbol{\theta}$ are by calculating

$$p_{\boldsymbol{\theta}}(\mathbf{x}) = \int_{\mathbb{R}^m} p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})\, p(\mathbf{z}) \, d\mathbf{z}$$

But how do we compute this integral? The $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$ is a neural network, so the integral has no closed form solution. There is no direct way of evaluating $p_{\boldsymbol{\theta}}(\mathbf{x})$, let alone maximising it.

But what if we can approximate it numerically? Consider picking $n$ values from each dimension of $\mathbf{z}$, producing the total of $n^m$ different $\mathbf{z}$. We feed each $\mathbf{z}$ through the decoder to obtain a $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$. We now have a rough approximation of $p_{\boldsymbol{\theta}}(\mathbf{x})$, right? The problem is that even for relatively small values of $n$ and $m$, say $n=100$ and $m=10$, we would have to run the decoder $100^{10}$ times, and that's only for a single training example! Okay, what if we do random Monte Carlo sampling $\mathbf{z} \sim p(\mathbf{z})$ $n$ times? Again, because the vast majority of the latent space produces garbage, most randomly sampled $\mathbf{z}$ will also produce garbage. Calculating $p_{\boldsymbol{\theta}}(\mathbf{x})$ that way is completely intractable. We need a different approach.

Instead of randomly searching a high-dimensional latent space hoping to hit a good $\mathbf{z}$, consider an encoder network $q_{\boldsymbol{\phi}}(\mathbf{z}\vert{}\mathbf{x})$, parameterized by $\boldsymbol{\phi}$. Given an $\mathbf{x}$, $q_{\boldsymbol{\phi}}(\mathbf{z}\vert{}\mathbf{x})$ outputs the parameters of the distribution which describes a region of the latent space that yields good reconstructions. For example, if the latent space is governed by a Gaussian distribution, the encoder outputs a mean $\boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x})$ and variance $\boldsymbol{\sigma}^2_{\boldsymbol{\phi}}(\mathbf{x})$. This distribution acts as an approximation to the true, intractable posterior $p_{\boldsymbol{\theta}}(\mathbf{z}\vert{}\mathbf{x})$, which cannot be directly computed because it requires $p_{\boldsymbol{\theta}}(\mathbf{x})$ since $p_{\boldsymbol{\theta}}(\mathbf{z}\vert{}\mathbf{x})=\frac{p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})p(\mathbf{z})}{p_{\boldsymbol{\theta}}(\mathbf{x})}$.

We now have a setup where $q_{\boldsymbol{\phi}}(\mathbf{z}\vert{}\mathbf{x})$ creates a distribution over $\mathbf{z}$, and the decoder $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$ can sample $\mathbf{z}$ from that distribution before reconstructing the image. Together, this framework is called a Variational Autoencoder (VAE). The word "variational" comes from Bayesian statistics, where one intractable distribution (in our case $p_{\boldsymbol{\theta}}(\mathbf{z}\vert{}\mathbf{x})$), is approximated by a tractable distribution (in our case a Gaussian $q_{\boldsymbol{\phi}}(\mathbf{z}\vert{}\mathbf{x})$).

![Basic VAE diagram](graphs/vae_diagram.png)

## 3. Evidence Lower Bound (ELBO)

But how do we train the VAE?

Given our dataset $\mathcal{D}$, we want to maximise

$$\sum_{i=1}^{n} \log p_{\theta}(\mathbf{x}_i)$$

But for now, let's focus on maximising the log likelihood for a single example $\log p_{\theta}(\mathbf{x})$. With our new encoder we can show that

$$
\begin{align*}
\log p_{\boldsymbol{\theta}}(\mathbf{x}) 
&= \log \int_{\mathbb{R}^m} p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z}) \, d\mathbf{z} \\[8pt]
&= \log \int_{\mathbb{R}^m} p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z}) \frac{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \, d\mathbf{z} \\[8pt]
&= \log \int_{\mathbb{R}^m} q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \frac{p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \, d\mathbf{z} \\[8pt]
&= \log \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \frac{p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \right]
\end{align*}
$$

We can now use Jensen's inequality to show that

$$
\begin{align*}
\log p_{\boldsymbol{\theta}}(\mathbf{x}) 
&\ge \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \right] \\[8pt]
&\ge \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \left( p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z}) \right) - \log q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \right] \\[8pt]
&\ge \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) + \log p(\mathbf{z}) - \log q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \right] \\[8pt]
&\ge \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right] + \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p(\mathbf{z}) - \log q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \right] \\[8pt]
&\ge \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right] + \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \frac{p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \right]
\end{align*}
$$

The second expectation equals $- D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right)$, so we can rewrite the final form as

$$
\begin{align*}
\log p_{\boldsymbol{\theta}}(\mathbf{x}) 
\ge \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right] - D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right)
\end{align*}
$$

On the right-hand side, we arrived at a tractable approximation of $\log p_{\boldsymbol{\theta}}(\mathbf{x})$, something that we can actually optimize in practice. The expectation term can now be approximated via Monte Carlo sampling, i.e. we sample $\log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z})$ many times and calculate an average, and the $D_{\text{KL}}$ term has a closed-form solution because both the prior and posterior distributions are Gaussian. It is customary that the intractable left-hand side is called an Evidence, and the tractable right-hand side is called an Evidence Lower Bound (**ELBO**).

Great! But what do we do with this ELBO? We maximise it. But, why? Consider this

$$\begin{align*}
D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})\right) 
&= \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \frac{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})}{p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})} \right] \\[8pt]
&= \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \left( \frac{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \cdot p_{\boldsymbol{\theta}}(\mathbf{x})}{p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})} \right) \right] \\[8pt]
&= \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x}) \right] + \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \frac{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})}{p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})} \right] \\[8pt]
&= \log p_{\boldsymbol{\theta}}(\mathbf{x}) - \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \right] \\[8pt]
&= \log p_{\boldsymbol{\theta}}(\mathbf{x}) - \text{ELBO}(\boldsymbol{\theta}, \boldsymbol{\phi}; \mathbf{x})
\end{align*}$$

rearranging, we can see that

$$
\log p_{\boldsymbol{\theta}}(\mathbf{x}) = \text{ELBO}(\boldsymbol{\theta}, \boldsymbol{\phi}; \mathbf{x}) + D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})\right) 
$$

Let's assume that the decoder $\log p_{\boldsymbol{\theta}}(\mathbf{x} | \mathbf{z})$ is frozen during training, and since the prior $p(\mathbf{z})$ is also fixed, the $\log p_{\boldsymbol{\theta}}(\mathbf{x})$ is a constant. Because $D_{\text{KL}} \ge 0$, as we increase $\text{ELBO}(\boldsymbol{\theta}, \boldsymbol{\phi}; \mathbf{x})$, the $D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})\right)$ term has to decrease for the equality to hold. In the limit, as $D_{\text{KL}} \to 0$, the lower bound eventually matches the evidence, that is $\log p_{\boldsymbol{\theta}}(\mathbf{x}) = \text{ELBO}(\boldsymbol{\theta}, \boldsymbol{\phi}; \mathbf{x})$. Think about what happened here. If the  $D_{\text{KL}} \approx 0$, that means $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \approx p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})$. What happens if we now freeze the encoder $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})$? Because the $D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right)$ becomes a constant, we can show that the ELBO gradient is equal to

$$\begin{align*}
\nabla_{\boldsymbol{\theta}} \text{ELBO}(\boldsymbol{\theta}, \boldsymbol{\phi}; \mathbf{x}) 
&= \nabla_{\boldsymbol{\theta}} \left( \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right] - D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right) \right) \\[8pt]
&= \nabla_{\boldsymbol{\theta}} \left( \int q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \, d\mathbf{z} \right) - \nabla_{\boldsymbol{\theta}} D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right) \\[8pt]
&= \int q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \, \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \, d\mathbf{z} - \mathbf{0} \\[8pt]
&= \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right]
\end{align*}$$

likewise, we can show the intractable gradient of $\log p_{\boldsymbol{\theta}}(\mathbf{x})$ to be equal to

$$\begin{align*}
\nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x}) 
&= \frac{\nabla_{\boldsymbol{\theta}} p_{\boldsymbol{\theta}}(\mathbf{x})}{p_{\boldsymbol{\theta}}(\mathbf{x})} \\[8pt]
&= \frac{1}{p_{\boldsymbol{\theta}}(\mathbf{x})} \nabla_{\boldsymbol{\theta}} \int p_{\boldsymbol{\theta}}(\mathbf{x}, \mathbf{z}) \, d\mathbf{z} \\[8pt]
&= \frac{1}{p_{\boldsymbol{\theta}}(\mathbf{x})} \int \nabla_{\boldsymbol{\theta}} p_{\boldsymbol{\theta}}(\mathbf{x}, \mathbf{z}) \, d\mathbf{z} \\[8pt]
&= \frac{1}{p_{\boldsymbol{\theta}}(\mathbf{x})} \int p_{\boldsymbol{\theta}}(\mathbf{x}, \mathbf{z}) \, \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x}, \mathbf{z}) \, d\mathbf{z} \\[8pt]
&= \int \frac{p_{\boldsymbol{\theta}}(\mathbf{x}, \mathbf{z})}{p_{\boldsymbol{\theta}}(\mathbf{x})} \, \nabla_{\boldsymbol{\theta}} \left( \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) + \log p(\mathbf{z}) \right) \, d\mathbf{z} \\[8pt]
&= \int p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x}) \left( \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) + \mathbf{0} \right) \, d\mathbf{z} \\[8pt]
&= \mathbb{E}_{p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})} \left[ \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right]
\end{align*}$$

because we know that $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \approx p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})$, this must mean that $\mathbb{E}_{p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})} \left[ \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right] \approx \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right]$. As a result, the direction that maximizes the ELBO is approximately also the direction that maximizes the intractable $\log p_{\boldsymbol{\theta}}(\mathbf{x})$.

Therefore, as both the decoder and encoder are active during training, by maximizing ELBO, the network does three things simultaneously:
- tries to make ELBO as close as possible to $\log p_{\boldsymbol{\theta}}(\mathbf{x})$
- tries to maximize $\log p_{\boldsymbol{\theta}}(\mathbf{x})$
- pushes the encoder distribution $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})$ closer to the true distribution $p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})$

The model is trying to maximise the reconstruction of $\mathbf{x}$, given that the $\mathbf{z}$ comes from the learned $q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})$ distribution. However, the $D_{\text{KL}}$ term ensures that $q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})$ does not drift away too far from the prior $p(\mathbf{z})$. For any input $\mathbf{x}$, both the true posterior $p_{\boldsymbol{\theta}}(\mathbf{z}\mid \mathbf{x})$ and the variational posterior $q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})$ are small, localized distributions (narrow bumps in the latent space). The $D_{\text{KL}}$ term forces every one of these localized distributions to reside within the high-density region of the prior $p(\mathbf{z})$.

Why is drifting away from the prior bad though? Remember that our goal is to maximize the evidence $\int p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z}) \, d\mathbf{z}$. If the decoder $\log p_{\boldsymbol{\theta}}(\mathbf{x} | \mathbf{z})$ is trained on $\mathbf{z}$'s that are far away from the prior $p(\mathbf{z})$, reusing that decoder in the evidence will produce garbage reconstructions because $\mathbf{z}$ is sampled from $p(\mathbf{z})$, a region of the latent space that the decoder was not trained on. That means all learned $q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})$ from across the dataset $\mathcal{D}$ will reside in the high-density region of the prior $p(\mathbf{z})=\mathcal{N}(\mathbf{0}, \mathbf{I})$. To generate a brand new $\mathbf{x}$, we can simply sample $\mathbf{z}$ from $p(\mathbf{z})$ and feed it through the decoder.

$$
\log p_{\boldsymbol{\theta}}(\mathbf{x}) \ge \boxed{\text{ELBO}(\mathbf{x}) = \underbrace{\mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})}\big[\log p_{\boldsymbol{\theta}}(\mathbf{x}\mid \mathbf{z})\big]}_{\text{expected reconstruction log-likelihood}} \;-\; \underbrace{D_{\text{KL}}\big(q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x}) \,\parallel\, p(\mathbf{z})\big)}_{\text{regularizes } q_{\boldsymbol{\phi}} \text{ toward the prior}}}
$$

Our final goal is therefore to maximize the sum of ELBOs from the training dataset $\mathcal{D}$

$$(\boldsymbol{\theta}^*, \boldsymbol{\phi}^*) = \arg\max_{\boldsymbol{\theta}, \boldsymbol{\phi}} \sum_{i=1}^{n} \text{ELBO}(\boldsymbol{\theta}, \boldsymbol{\phi}; \mathbf{x}_i)$$


## 4. Reparametrization trick

Okay, let's whip out PyTorch and train the model then. We can skip the details of the loss function for now and just assume that $\mathcal{L}=-\text{ELBO}$. There is a problem. How do we backpropagate the error through the sampling process $\mathbf{z} \sim q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})$, i.e. how do we connect the encoder and decoder in the computational graph? Our encoder posterior is governed by a multivariate Gaussian distribution $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}\left(\boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}), \boldsymbol{\Sigma}_{\boldsymbol{\phi}}(\mathbf{x})\right)$, where $\boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x})$ is of shape $(m, 1)$, and the covariance $\boldsymbol{\Sigma}_{\boldsymbol{\phi}}(\mathbf{x})$ is of shape $(m, m)$. However, the original VAE assumes that the latent space dimensions are completely independent, so the covariance ends up being a diagonal matrix, where $\boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})$ is of shape $(m, 1)$

$$\boldsymbol{\Sigma}_{\boldsymbol{\phi}}(\mathbf{x}) = \operatorname{diag}\left(\boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})\right) = \begin{bmatrix} \sigma_{\boldsymbol{\phi}, 1}^2(\mathbf{x}) & 0 & \cdots & 0 \\ 0 & \sigma_{\boldsymbol{\phi}, 2}^2(\mathbf{x}) & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & \sigma_{\boldsymbol{\phi}, m}^2(\mathbf{x}) \end{bmatrix}$$

However, variance has to be non-negative, and standard neurons can output any value from the range $(-\infty, \infty)$. We have to somehow force the network to output non-negative variance only. You might think just using $\text{RELU}$ is good enough to force non-negativity, but that actually crashes the training, but let's ignore that for now. In practice, we force the encoder to output $\log \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})$ where any value from $\mathbb{R}$ is allowed, and whenever we need to use the actual variance, we do $e^{\log \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})}$. That way, we get the variance back, since $\mathbf{e}^{\log \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})}= \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})$, but we also enforce non-negativity, since $\mathbf{e}^{\log \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})} \ge 0$. So the final form of the encoder posterior is $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}\left( \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}), \, \operatorname{diag}\left(\exp\left(\log\boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})\right)\right)\right)$.

Now, back to the main problem, how to connect the encoder and decoder. Consider a new random variable $\boldsymbol{\epsilon} \in \mathbb{R}^m$, where $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$. The sampling from $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})$ is mathematically equivalent to

$$\mathbf{z} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}) + \mathbf{s}_{\boldsymbol{\phi}}(\mathbf{x}) \odot \boldsymbol{\epsilon}$$

where $\mathbf{s}$ is the standard deviation and is equal to $\mathbf{s}_{\boldsymbol{\phi}}(\mathbf{x})=\sqrt{\mathbf{e}^{\log \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})}}= \mathbf{e}^{\frac{1}{2}\log \sigma_{\boldsymbol{\phi}}^2(\mathbf{x})}$. Why is that a mathematically sound sample from the $q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})$? Let's consider the individual dimensions of this new $\mathbf{z}$ separately, i.e. $1$ to $m$. We will use these two properties of random variables
- *Linearity of Expectation*: $\mathbb{E}[a + bX] = a + b\mathbb{E}[X]$
- *Scaling property of Variance*: $\operatorname{Var}(a + bX) = b^2 \operatorname{Var}(X)$

Now, assuming $j$ is one of the dimensions of $\mathbf{z}$, let's calculate its mean and variance given that the source of randomness is this $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$.

$$
\begin{aligned}
\mathbb{E}[z_j] &= \mathbb{E}\left[ \mu_{\boldsymbol{\phi}, j}(\mathbf{x}) + \sqrt{e^{\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})}} \cdot \epsilon_j \right] \\
&= \mu_{\boldsymbol{\phi}, j}(\mathbf{x}) + \sqrt{e^{\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})}} \cdot \mathbb{E}[\epsilon_j] \\
&= \mu_{\boldsymbol{\phi}, j}(\mathbf{x}) + \sqrt{e^{\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})}} \cdot 0 \\
&= \mu_{\boldsymbol{\phi}, j}(\mathbf{x})
\end{aligned}
$$

$$
\begin{aligned}
\operatorname{Var}(z_j) &= \operatorname{Var}\left( \mu_{\boldsymbol{\phi}, j}(\mathbf{x}) + \sqrt{e^{\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})}} \cdot \epsilon_j \right) \\
&= \left( \sqrt{e^{\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})}} \right)^2 \cdot \operatorname{Var}(\epsilon_j) \\
&= e^{\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})} \cdot 1 \\
&= \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})
\end{aligned}
$$

Because the dimensions of the latent space are independent, the mean and variance of the full new $\mathbf{z}$ are

$$\mathbb{E}[\mathbf{z}] = \begin{bmatrix} \mathbb{E}[z_1] \\ \mathbb{E}[z_2] \\ \vdots \\ \mathbb{E}[z_m] \end{bmatrix} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x})$$

$$\operatorname{Var}(\mathbf{z}) = \operatorname{diag}\left( \operatorname{Var}(z_1), \operatorname{Var}(z_2), \dots, \operatorname{Var}(z_m) \right) = \operatorname{diag}\left( \boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x}) \right)$$

This operation $\mathbf{z} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}) + \mathbf{s}_{\boldsymbol{\phi}}(\mathbf{x}) \odot \boldsymbol{\epsilon}$, is equivalent to a linear transformation of $\boldsymbol{\epsilon}$. Since $\boldsymbol{\epsilon}$ is a Gaussian, the linearly transformed $\boldsymbol{\epsilon}$ is also a Gaussian but with a different mean and variance. However, we proved that this new mean and variance exactly match the mean and variance of the posterior $q_{\boldsymbol{\phi}}(\mathbf{z}\mid \mathbf{x})$. That means sampling $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and computing $\mathbf{z} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}) + \sqrt{\exp\left(\log \boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})\right)} \odot \boldsymbol{\epsilon}$ is identically equivalent to sampling $\mathbf{z} \sim \mathcal{N}\left(\boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}), \operatorname{diag}\left(\exp\left(\log \boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})\right)\right)\right)$ directly.

Nice! But why does that matter? It matters because $\mathbf{z} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}) + \mathbf{s}_{\boldsymbol{\phi}}(\mathbf{x}) \odot \boldsymbol{\epsilon}$ can now be part of the computational graph. The problem of backpropagating through the sampling process has been moved to $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, allowing the gradient to flow through $\frac{\partial \mathcal{L}}{\partial \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x})}$ and $\frac{\partial \mathcal{L}}{\partial \log \boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})}$ from the decoder to the encoder.

![VAE diagram with the reparameterization trick](graphs/vae_diagram_reparameterized.png)

## 5. Loss function

Nice. We're finally ready to jump into the details of how ELBO is calculated in practice.

$$
\text{ELBO}(\mathbf{x}) = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})} \left[ \log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) \right] - D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right)
$$

Starting with the $ D_{\text{KL}}$ term because the approach is simple here, we have a closed-form solution! Since both distributions are Gaussian, the divergence simplifies to

$$
D_{\text{KL}}\left(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})\right) = -\frac{1}{2} \sum_{j=1}^{m} \left( 1 + \log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x}) - \mu_{\boldsymbol{\phi}, j}^2(\mathbf{x}) - \exp\left(\log \sigma_{\boldsymbol{\phi}, j}^2(\mathbf{x})\right) \right)
$$

Now, how do we calculate the expectation term? We know that Monte Carlo sampling tells us that

$$
\mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z})\right] = \lim_{L \to \infty} \frac{1}{L} \sum_{l=1}^{L} \log p_{\boldsymbol{\theta}}\left(\mathbf{x} \mid \mathbf{z}^{(l)}\right)
$$

which means that a single reconstruction from one $\mathbf{z}$ that we get from a forward pass, can be used as a rough approximation

$$
\mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x})}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z})\right] \approx \log p_{\boldsymbol{\theta}}\left(\mathbf{x} \mid \mathbf{z}\right)
$$

Nice. Now the question is how do we calculate $\log p_{\boldsymbol{\theta}}\left(\mathbf{x} \mid \mathbf{z}\right)$? This is where we can use the standard loss functions that you're probably familiar with, CCE, BCE, MSE, MAE etc, the choice is yours depending on the underlying data.

Assume that $\mathbf{x}$ is a normalized $H \times W$ image, that is, every pixel $x_{i,j}$ is divided by $255$ so that its value lies in the range $[0,1]$. Given that the decoder spits out a reconstruction $\mathbf{\hat{x}}$, our goal is $\mathbf{\hat{x}}_{i,j}=\mathbf{x}_{i,j}$ for all $i$ and $j$. 

Let's choose the Binary Cross-Entropy loss here because it ties nicely to our derivation of KL divergence, and is also the most commonly used in practice. Remember, our definition of Cross-Entropy is

$$H(P, Q)=\sum_{i=1}^{n} P(x_i) \log \left( \frac{1}{Q(x_i)} \right)$$

If we apply $H(P, Q)$ to an individual pixel $x_{i,j}$, we can consider a pixel-level reconstruction loss to be

$$
\mathcal{L}_{\text{BCE}}(x_{i,j}, \hat{x}_{i,j}) = -\left[ x_{i,j} \log \hat{x}_{i,j} + (1 - x_{i,j}) \log(1 - \hat{x}_{i,j}) \right]
$$

You should also remember that the minimum value of $\mathcal{L}_{\text{BCE}}(x_{i,j}, \hat{x}_{i,j})$ is reached when $\hat{x}_{i,j}=x_{i,j}$. That means

$$\log p_{\boldsymbol{\theta}}\left(\mathbf{x} \mid \mathbf{z}\right) = -\sum_{i=1}^{H} \sum_{j=1}^{W} \left[ x_{i,j} \log \hat{x}_{i,j} + (1 - x_{i,j}) \log(1 - \hat{x}_{i,j}) \right]$$

And that is the VAE done! To train via gradient descent, we minimize $\mathcal{L}_{\text{VAE}} = -\text{ELBO}$.